# Mobile Price Predictions



In this notebook, we will demonstrate how the Mobile Price Classification example used in our examples could be implemented in plain python. Dataset available [on Kaggle](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification?datasetId=11167&sortBy=voteCount).

> **⚠️ Note:** The hyperparameter tuning grid in this notebook is intentionally much smaller than in the pipeline examples. Notebooks run on limited resources and are not suited for heavy compute workloads. For high-performance workloads, always use a **Kubeflow Pipeline**, which can distribute work across the cluster and scale far beyond what a single notebook can handle.

In [ ]:
%pip -q install kagglehub plotly

In [ ]:
import os
import pandas as pd
import numpy as np
from typing import Dict, Tuple, List
import s3fs

In [ ]:
# Configuration for MinIO bucket
# Change to your MinIO bucket name, which is <namespace>-data by default
# You can check your buckets by opening a terminal in the notebook and run 'mc ls minio'
s3_bucket = ""
s3_dataset_path = f"{s3_bucket}/mobile-price-classification"
if not s3_bucket:
    raise ValueError("Please set the 's3_bucket' variable to your MinIO bucket name.")

minio_train_data_path=f"s3://{s3_dataset_path}/train.csv"
minio_test_data_path=f"s3://{s3_dataset_path}/test.csv"

## Define some constants for later

Below we define arguments used throughout the pipeline. **`test_size`** controls what fraction of the training data is held out for validation. **`C`**, **`kernel`**, **`gamma`**, and **`decision_function_shape`** are hyperparameter grids for the SVM classifier. **`scatter_plot_column_x`** and **`scatter_plot_column_y`** specify which features to use for visualization scatter plots. Finally, **`seed`** ensures reproducibility across runs.

In [ ]:
test_size=0.2
C=[0.1,1]
kernel=["linear"]
gamma=["auto", 0.01 ]
decision_function_shape=["ovo"]
scatter_plot_column_x='ram'
scatter_plot_column_y='battery_power'
seed=42

## Download Dataset From Kagglehub and Upload to MinIO with Python

In [ ]:
# Download Dataset From Kagglehub
import kagglehub
dataset_path = kagglehub.dataset_download("iabhishekofficial/mobile-price-classification")
print(f"Kaggle dataset downloaded to: {dataset_path}")

In [ ]:
# Upload dataset to MinIO
if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
    raise ValueError("AWS credentials not found in environment variables.")

# Initialize S3 filesystem
s3 = s3fs.S3FileSystem()

# Upload the dataset to MinIO
s3.put(f"{dataset_path}/train.csv", f"{s3_dataset_path}/train.csv")
s3.put(f"{dataset_path}/test.csv", f"{s3_dataset_path}/test.csv")

print(s3.ls(s3_dataset_path))

## One cell for each component

### Read data

In [ ]:
# The storage options are preconfigured for the cluster internal MinIO, so we can read the data directly without additional configuration
df_train = pd.read_csv(minio_train_data_path)
df_test = pd.read_csv(minio_test_data_path)

### Split data

In [ ]:
from sklearn.model_selection import train_test_split 

# Separate target from features
y = df_train["price_range"].to_frame()
x_data = df_train.drop(["price_range"], axis=1)
    
# Split the data into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(x_data, y, test_size=test_size, random_state=seed)

### Fit scaler

In [ ]:
"""
Fits a MinMaxScaler on the provided training data and saves the fitted scaler.
"""
from sklearn.preprocessing import MinMaxScaler
    
# Fit the MinMaxScaler on the training data
scaler = MinMaxScaler()
scaler.fit(x_train)

### Run grid search

In [ ]:
"""
Performs hyperparameter tuning using GridSearchCV for a SVM classifier on the provided training data.
Returns the best hyperparameters found.
"""
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Initialize SVM with a random seed
svm = SVC(random_state=seed)

# Define grid search with provided hyperparameters
grid_svm = GridSearchCV(
    estimator=svm,
    cv=5,
    param_grid=dict(
        kernel=kernel, 
        C=C, 
        gamma=gamma, 
        decision_function_shape=decision_function_shape
    ),
    n_jobs=-1,
    verbose=2
)

# Perform grid search
grid_svm.fit(x_train, y_train['price_range'].values)
    
# Print the best score found
print("Best score: ", grid_svm.best_score_)

    # Return the best hyperparameters
hparams = grid_svm.best_params_

### Train model with optimal hyper parameters

In [ ]:
"""
Trains an SVM classifier on the provided training data using the best hyperparameters from tuning.
The trained model is then saved.
"""
from sklearn.svm import SVC

# Read and preprocess the training data
x_train = pd.DataFrame(scaler.transform(x_train), columns=x_train.columns)

# Initialize SVM with the best hyperparameters and a random seed
svm_model = SVC(random_state=seed, **hparams)

# Train the SVM model
svm_model.fit(x_train, y_train['price_range'].values)

### Evaluate model on validation dataset

In [ ]:
"""
Evaluates the performance of a trained SVM model using validation data.
Outputs a confusion matrix plot and a markdown file containing a classification report.
"""
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report

x_val = pd.DataFrame(scaler.transform(x_val), columns=x_val.columns)

predictions = svm_model.predict(x_val)

confusion_matrix(y_val['price_range'].values, predictions)

In [ ]:
# The model is not performing well, so we set zero_division=0 to avoid errors in the classification report. See note above.
print(classification_report(y_val['price_range'].values, predictions, zero_division=0))

### Run predictions on test dataset

In [ ]:
"""
Test a trained SVM model on provided test data and produce a scatter plot.
The scatter plot will have points colored by the predicted class based on two columns
specified by the user.
"""
import plotly.express as px

# Load the fitted scaler and preprocess test data
x_test = pd.read_csv(minio_test_data_path)
x_test = x_test.drop('id', axis=1)
x_test = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns)

# Load the trained SVM model and make predictions on test data
predictions = svm_model.predict(x_test)

# Add predictions as a column to the x_test DataFrame for visualization
x_test['Predicted Class'] = predictions

# Create the scatter plot using plotly
fig = px.scatter(x_test,
                    x=scatter_plot_column_x,
                    y=scatter_plot_column_y,
                    color='Predicted Class',
                    color_continuous_scale='Viridis',
                    title=f"Scatter plot of {scatter_plot_column_x} vs. {scatter_plot_column_y} colored by Predicted Class",
                    template='plotly_dark')
fig.show()